
# Partie A — Quiver Congress Trading → `final_table` commentée

## Réponse courte à la question : est-ce que l'endpoint Quiver suffit ?

Pour **la Partie A simple**, oui : l'endpoint Quiver suivant suffit :

```text
GET https://api.quiverquant.com/beta/bulk/congresstrading
```

Il sert à récupérer l'historique des transactions déclarées par les membres du Congrès. C'est donc la bonne source pour construire une première table propre à partir de Quiver.

Ce notebook fait volontairement uniquement ceci :

```text
API Quiver
    ↓
données Congress Trading brutes
    ↓
nettoyage minimal
    ↓
normalisation des colonnes
    ↓
une seule table finale : final_table
```

Il ne fait pas encore :

```text
- récupération des prix de marché ;
- calcul de rendement ;
- backtest ;
- mapping ticker → secteur GICS ;
- mapping secteur → ETF ;
- ajout des commissions ;
- sélection annuelle des congressmen.
```

Donc :

```text
Oui, cet endpoint suffit pour la récupération Quiver de base.
Non, il ne suffit pas pour tout le projet final, car le backtest aura aussi besoin des prix,
des secteurs, des ETF proxies et des commissions.
```



# Logique générale du notebook

Le notebook est découpé en 6 blocs :

```text
1. Configuration
2. Imports
3. Client API Quiver
4. Helpers de nettoyage
5. Normalisation Quiver → table finale
6. Exécution : création de final_table
```

Le point le plus important à retenir est la distinction entre les deux dates :

```text
transaction_date = date réelle du trade effectué par l'élu

disclosure_date = date à laquelle ce trade devient public
```

Pour la stratégie, on ne pourra pas entrer à `transaction_date`, car l'information n'était pas encore publique. On devra entrer à partir de `disclosure_date` ou du premier jour de marché suivant. Cette Partie A garde donc les deux dates et calcule :

```text
disclosure_lag_days = disclosure_date - transaction_date
```

Cette colonne servira dans la Partie B pour mesurer combien de jours se sont écoulés entre le trade réel et la divulgation publique.


In [1]:

# ============================================================
# 1) CONFIGURATION — tout est ici, rien en variable d'environnement
# ============================================================

# Colle ton token Quiver ici.
# Important : ne mets pas ce notebook sur GitHub avec le token rempli.
# Si le token est déjà apparu quelque part, il vaut mieux le régénérer dans Quiver.
QUIVER_API_TOKEN = "e4f3cb20daf202dfc7bfae07af4ee8726a78affe"  # <- exemple : "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

# URL de base de l'API Quiver.
BASE_URL = "https://api.quiverquant.com"

# Endpoint utilisé.
# C'est l'endpoint "Bulk Congress Trading" de Quiver.
# Il renvoie l'historique des transactions des membres du Congrès.
ENDPOINT = "/beta/bulk/congresstrading"

# Paramètres envoyés à Quiver.
PARAMS = {
    # normalized=true : demande à Quiver de normaliser les noms des élus.
    # Exemple : éviter que le même élu apparaisse sous plusieurs variantes de nom.
    "normalized": "true",

    # nonstock=false : on ne veut pas inclure les transactions non-actions dans cette V0.
    # Le projet commence par les trades actions / stock-like.
    # Les options, obligations, fonds complexes, etc. peuvent venir plus tard.
    "nonstock": "false",

    # version=V2 : on utilise le schéma récent de Quiver.
    # V2 donne notamment :
    # - Name
    # - BioGuideID
    # - Chamber
    # - Party
    # - Ticker
    # - TickerType
    # - Company
    # - Traded      -> transaction_date
    # - Filed       -> disclosure_date
    # - Transaction -> Purchase / Sale
    # - Trade_Size_USD
    "version": "V2",
}

# Quiver renvoie les résultats par pages.
# PAGE_SIZE = nombre maximal de lignes par page.
PAGE_SIZE = 1000

# Pour tester vite sans télécharger tout l'historique, mets par exemple MAX_PAGES = 2.
# Pour tout récupérer, laisse MAX_PAGES = None.
MAX_PAGES = None

# Pause entre deux appels API, pour éviter d'enchaîner les requêtes trop vite.
SLEEP_SECONDS = 0.25

# Timeout d'un appel API.
TIMEOUT_SECONDS = 60

# Nombre maximal de tentatives en cas d'erreur temporaire.
MAX_RETRIES = 5

# Pour respecter l'idée "une table finale c'est tout", on n'affiche pas
# les logs de pagination par défaut. Mets True si tu veux voir page par page.
SHOW_PROGRESS = False

# Filtres appliqués à la fin.
# Ils servent à produire une table directement exploitable pour la suite.
DROP_MISSING_TICKER = True       # enlever les lignes sans ticker
DROP_MISSING_DATES = True        # enlever les lignes sans transaction_date ou disclosure_date
KEEP_ONLY_PURCHASE_SALE = True   # garder seulement Purchase / Sale

# Convention pour les tranches ouvertes du type "> $50,000,000".
# Une tranche ouverte n'a pas de vraie borne haute, donc pas de vrai midpoint.
# Cette valeur est une convention documentée, pas une vérité économique.
OPEN_RANGE_MIDPOINT = 50_000_000.0



## Étape 1 — Configuration

Cette cellule fixe tout ce que le notebook doit savoir avant de tourner.

Les éléments importants sont :

```text
QUIVER_API_TOKEN
```

C'est ton token Quiver. Le notebook l'utilise pour s'authentifier auprès de l'API.

```text
ENDPOINT = "/beta/bulk/congresstrading"
```

C'est l'endpoint principal. Il évite de faire des appels ticker par ticker. On récupère directement l'historique Congress Trading.

```text
PARAMS = {"normalized": "true", "nonstock": "false", "version": "V2"}
```

Ces paramètres veulent dire :

```text
normalized=true
→ demander à Quiver de normaliser les noms des élus.

nonstock=false
→ ne pas inclure les transactions non-actions dans cette première version.

version=V2
→ utiliser le schéma récent avec les colonnes Traded et Filed.
```

Les paramètres `PAGE_SIZE` et `MAX_PAGES` contrôlent la pagination :

```text
PAGE_SIZE = 1000
→ on demande 1000 transactions par page.

MAX_PAGES = None
→ on récupère toutes les pages.

MAX_PAGES = 2
→ on récupère seulement deux pages pour tester vite.
```


In [2]:

# ============================================================
# 2) IMPORTS
# ============================================================

# hashlib : sert à créer un identifiant unique stable pour chaque transaction.
import hashlib

# math : utilisé pour vérifier certains nombres dans le parsing des montants.
import math

# re : expressions régulières, utilisées pour extraire les montants dans du texte.
import re

# time : pauses entre les appels API et attente en cas de rate limit.
import time

# Any : annotation de type pour les réponses API.
from typing import Any

# numpy / pandas : manipulation des tables et valeurs manquantes.
import numpy as np
import pandas as pd

# requests : appels HTTP vers l'API Quiver.
import requests

# display : affichage propre de final_table dans Jupyter.
from IPython.display import display

# Options d'affichage pandas pour voir davantage de colonnes.
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)



## Étape 2 — Imports

Cette cellule ne fait aucune logique métier.

Elle charge seulement les librairies nécessaires :

```text
requests
→ appeler l'API Quiver.

pandas
→ manipuler les tables.

numpy
→ gérer les valeurs numériques manquantes.

re
→ extraire les montants dans les chaînes de caractères.

hashlib
→ créer un trade_id unique.
```



## Étape 3 — Client API Quiver

Cette partie contient les fonctions qui appellent Quiver.

Il y a trois fonctions :

```text
_extract_rows(payload)
```

Elle récupère la liste de transactions à partir de la réponse JSON. Normalement, Quiver renvoie directement une liste. Cette fonction rend le notebook un peu plus robuste si la réponse est encapsulée dans un dictionnaire.

```text
quiver_get(endpoint, params)
```

Elle fait un appel API réel. Elle ajoute le token dans le header :

```text
Authorization: Bearer <token>
```

Elle gère aussi les erreurs classiques :

```text
401 / 403 → problème d'authentification ou d'abonnement
429       → trop de requêtes, donc on attend puis on réessaie
```

```text
fetch_all_congress_trades()
```

Elle télécharge toutes les pages une par une.

Exemple de pagination :

```text
page 1 → 1000 lignes
page 2 → 1000 lignes
page 3 → 1000 lignes
page 4 → 417 lignes
```

Comme la page 4 contient moins que `PAGE_SIZE`, le notebook comprend qu'il est arrivé à la fin.

À la fin de cette étape, on obtient une table brute :

```text
raw_quiver_table
```


In [3]:

# ============================================================
# 3) CLIENT API QUIVER
# ============================================================


def _extract_rows(payload: Any) -> list[dict]:
    """
    Rôle : extraire la liste de transactions depuis la réponse JSON Quiver.

    Dans le cas attendu, Quiver renvoie directement :

        [ {...}, {...}, {...} ]

    Mais certaines API renvoient parfois :

        {"results": [...]}
        {"data": [...]}
        {"items": [...]}

    Cette fonction accepte ces formats par prudence.
    """
    if payload is None:
        return []

    if isinstance(payload, list):
        return payload

    if isinstance(payload, dict):
        for key in ("results", "data", "items"):
            value = payload.get(key)
            if isinstance(value, list):
                return value

        # Cas plus imbriqué possible : {"data": {"results": [...]}}
        data = payload.get("data")
        if isinstance(data, dict):
            for key in ("results", "items"):
                value = data.get(key)
                if isinstance(value, list):
                    return value

    raise TypeError(f"Format de réponse Quiver inattendu : {type(payload)}")


def quiver_get(endpoint: str, params: dict | None = None) -> list[dict]:
    """
    Rôle : faire un appel GET à Quiver.

    Cette fonction :
    1. vérifie que le token est rempli ;
    2. construit l'URL complète ;
    3. ajoute l'authentification Bearer ;
    4. appelle l'API ;
    5. gère les erreurs ;
    6. renvoie une liste de lignes JSON.
    """
    if not QUIVER_API_TOKEN or QUIVER_API_TOKEN.strip() == "":
        raise ValueError("Colle d'abord ton token Quiver dans QUIVER_API_TOKEN.")

    url = f"{BASE_URL}{endpoint}"
    headers = {"Authorization": f"Bearer {QUIVER_API_TOKEN.strip()}"}

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(
                url,
                headers=headers,
                params=params,
                timeout=TIMEOUT_SECONDS,
            )

            # 429 = rate limit. On attend puis on réessaie.
            if response.status_code == 429:
                wait = min(60, 2 ** attempt)
                if SHOW_PROGRESS:
                    print(f"Rate limit Quiver 429. Attente {wait}s puis nouvel essai...")
                time.sleep(wait)
                continue

            # 401 / 403 = problème d'accès.
            if response.status_code in (401, 403):
                raise RuntimeError(
                    f"Erreur auth Quiver {response.status_code}. "
                    "Vérifie le token et le plan d'abonnement."
                )

            response.raise_for_status()
            return _extract_rows(response.json())

        except Exception as exc:
            last_error = exc
            if attempt == MAX_RETRIES:
                break
            wait = min(60, 2 ** attempt)
            if SHOW_PROGRESS:
                print(f"Erreur API temporaire : {exc}. Attente {wait}s puis nouvel essai...")
            time.sleep(wait)

    raise RuntimeError(f"Échec API Quiver après {MAX_RETRIES} essais : {last_error}")


def fetch_all_congress_trades() -> pd.DataFrame:
    """
    Rôle : télécharger toutes les pages Quiver Congress Trading.

    Cette fonction ne sauvegarde rien sur disque.
    Elle retourne simplement un DataFrame brut : raw_quiver_table.
    """
    rows_all: list[dict] = []
    page = 1

    while True:
        # On part des paramètres généraux, puis on ajoute la page courante.
        params = dict(PARAMS)
        params.update({"page": page, "page_size": PAGE_SIZE})

        rows = quiver_get(ENDPOINT, params=params)
        rows_all.extend(rows)

        if SHOW_PROGRESS:
            print(f"page={page} | lignes reçues={len(rows)} | total={len(rows_all)}")

        # Si Quiver renvoie moins que PAGE_SIZE, on est probablement à la dernière page.
        if len(rows) < PAGE_SIZE:
            break

        # Option de test rapide : s'arrêter après MAX_PAGES pages.
        if MAX_PAGES is not None and page >= MAX_PAGES:
            break

        page += 1
        time.sleep(SLEEP_SECONDS)

    return pd.DataFrame(rows_all)



## Étape 4 — Helpers de normalisation

Ces fonctions ne téléchargent rien. Elles servent seulement à nettoyer les données.

Elles répondent à cinq problèmes pratiques.

### 1. Quiver peut utiliser plusieurs noms de colonnes

En V2, le nom du déclarant est plutôt :

```text
Name
```

En V1, il peut être :

```text
Representative
Senator
```

La fonction `first_existing` cherche la première colonne disponible.

### 2. Les dates doivent être converties proprement

Quiver peut donner des dates au format texte. La fonction `parse_date_series` les convertit en vraies dates pandas, normalisées au jour.

### 3. Les tickers doivent être propres

Exemples :

```text
" aapl "  → "AAPL"
"brk/b"   → "BRK.B"
"N/A"     → valeur manquante
```

### 4. Les opérations doivent être simplifiées

On veut surtout :

```text
Purchase
Sale
```

La fonction `normalize_operation` transforme les variantes possibles en catégories simples.

### 5. Les montants sont déclarés par fourchettes

Exemple :

```text
$1,001 - $15,000
```

Le notebook calcule :

```text
amount_lower_bound = 1001
amount_upper_bound = 15000
amount_midpoint    = 8000.5
amount_method      = "range_midpoint"
```

Si Quiver donne seulement une borne basse comme `1001`, le notebook met :

```text
amount_method = "lower_bound_only"
```

C'est important : dans ce cas, `amount_midpoint` n'est pas un vrai midpoint, c'est seulement la meilleure valeur disponible dans cette V0.


In [4]:

# ============================================================
# 4) HELPERS DE NORMALISATION
# ============================================================


def first_existing(df: pd.DataFrame, candidates: list[str], default=None):
    """
    Retourne la première colonne existante parmi candidates.

    Exemple :
        first_existing(df, ["Name", "Representative", "Senator"])

    Si df contient "Name", on renvoie df["Name"].
    Sinon, on cherche "Representative".
    Sinon, on cherche "Senator".
    Si aucune colonne n'existe, on renvoie une Series remplie avec default.
    """
    for col in candidates:
        if col in df.columns:
            return df[col]
    return pd.Series([default] * len(df), index=df.index)


def parse_date_series(s: pd.Series) -> pd.Series:
    """
    Convertit une série de dates Quiver en dates pandas.

    On normalise au jour parce que la stratégie V0 raisonne d'abord en dates,
    pas en timestamps intraday.

    Exemple :
        "2026-05-18T00:00:00Z" -> 2026-05-18
    """
    dt = pd.to_datetime(s, errors="coerce", utc=True)
    try:
        dt = dt.dt.tz_convert(None)
    except Exception:
        try:
            dt = dt.dt.tz_localize(None)
        except Exception:
            pass
    return dt.dt.normalize()


def clean_text_series(s: pd.Series) -> pd.Series:
    """
    Nettoie une colonne de texte :
    - transforme en string pandas ;
    - remplace plusieurs espaces par un seul ;
    - enlève les espaces au début et à la fin ;
    - transforme les chaînes vides en valeurs manquantes.
    """
    return (
        s.astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "NaT": pd.NA})
    )


def clean_ticker_series(s: pd.Series) -> pd.Series:
    """
    Nettoie les tickers.

    Exemples :
        " aapl "  -> "AAPL"
        "brk/b"   -> "BRK.B"
        "N/A"     -> NA

    Le remplacement de "/" par "." sert notamment pour des tickers du type BRK/B.
    """
    out = clean_text_series(s).str.upper()
    out = out.str.replace(" ", "", regex=False)
    out = out.str.replace("/", ".", regex=False)
    out = out.replace({"-": pd.NA, "—": pd.NA, "N/A": pd.NA, "NA": pd.NA, "NONE": pd.NA})
    return out


def normalize_operation(x) -> str:
    """
    Ramène les opérations à quelques catégories simples.

    Pour la stratégie, les deux catégories centrales sont :
        Purchase -> signal d'entrée
        Sale     -> signal de sortie
    """
    if pd.isna(x):
        return "Other"

    t = str(x).strip().lower()

    if t in {"p", "purchase", "purchased", "buy", "bought"} or "purchase" in t:
        return "Purchase"

    if (
        t in {"s", "sale", "sold", "sell"}
        or "sale" in t
        or "sold" in t
        or "sell" in t
    ):
        return "Sale"

    if "exchange" in t:
        return "Exchange"

    return "Other"


def normalize_asset_type(x) -> str:
    """
    Simplifie le type d'actif.

    Exemples :
        ST / CS / Common Stock -> Stock
        Option                 -> Option
        ETF / Fund / OT        -> Fund/ETF
    """
    if pd.isna(x):
        return "Unknown"

    t = str(x).strip().lower()

    if t in {"st", "cs"}:
        return "Stock"
    if "common stock" in t or t == "stock":
        return "Stock"
    if "option" in t:
        return "Option"
    if "etf" in t or "fund" in t or t in {"ot"}:
        return "Fund/ETF"
    if "bond" in t or "note" in t:
        return "Bond"

    return "Other"


def money_to_float(x) -> float | None:
    """
    Extrait le premier nombre d'une chaîne monétaire.

    Exemple :
        "$1,001" -> 1001.0
    """
    if x is None or pd.isna(x):
        return None

    s = str(x)
    s = s.replace("$", "").replace(",", "").replace("USD", "")
    s = s.replace("usd", "").replace("+", "")
    s = s.strip()

    if s == "":
        return None

    m = re.search(r"[-+]?\d*\.?\d+", s)
    if not m:
        return None

    try:
        return float(m.group(0))
    except Exception:
        return None


def parse_amount_range(x) -> pd.Series:
    """
    Parse les montants Quiver/PTR.

    Cas possibles :
        "$1,001 - $15,000"
        "$1,001 to $15,000"
        "> $50,000,000"
        "$50,000,000+"
        "1001" ou "$1001" si Quiver ne donne qu'une borne basse

    Le résultat contient quatre colonnes :
        amount_lower_bound
        amount_upper_bound
        amount_midpoint
        amount_method
    """
    if x is None or pd.isna(x):
        return pd.Series({
            "amount_lower_bound": np.nan,
            "amount_upper_bound": np.nan,
            "amount_midpoint": np.nan,
            "amount_method": "missing",
        })

    raw = str(x).strip()
    s = raw.lower().replace("\u2013", "-").replace("\u2014", "-").replace("–", "-").replace("—", "-")

    is_open = any(token in s for token in [">", "over", "more than"]) or s.strip().endswith("+")

    nums = re.findall(r"\$?\s*[\d,]+(?:\.\d+)?", raw)
    values = [money_to_float(n) for n in nums]
    values = [v for v in values if v is not None and not math.isnan(v)]

    # Cas classique : fourchette avec deux bornes.
    if len(values) >= 2:
        lo, hi = min(values[0], values[1]), max(values[0], values[1])
        return pd.Series({
            "amount_lower_bound": lo,
            "amount_upper_bound": hi,
            "amount_midpoint": (lo + hi) / 2,
            "amount_method": "range_midpoint",
        })

    # Cas avec une seule valeur.
    if len(values) == 1:
        lo = values[0]

        # Tranche ouverte : > 50M par exemple.
        # On utilise une convention explicite.
        if is_open:
            return pd.Series({
                "amount_lower_bound": lo,
                "amount_upper_bound": np.nan,
                "amount_midpoint": OPEN_RANGE_MIDPOINT,
                "amount_method": "open_range_convention",
            })

        # Si Quiver donne seulement une borne basse, on la garde.
        # Ce n'est pas un vrai midpoint.
        return pd.Series({
            "amount_lower_bound": lo,
            "amount_upper_bound": np.nan,
            "amount_midpoint": lo,
            "amount_method": "lower_bound_only",
        })

    return pd.Series({
        "amount_lower_bound": np.nan,
        "amount_upper_bound": np.nan,
        "amount_midpoint": np.nan,
        "amount_method": "unparsed",
    })


def make_trade_id(row: pd.Series) -> str:
    """
    Crée un identifiant interne de transaction.

    Pourquoi ?
    Quiver ne garantit pas forcément un identifiant unique stable pour chaque ligne.
    On crée donc un hash à partir des champs qui décrivent la transaction.

    Ce trade_id sert ensuite à dédupliquer.
    """
    cols = [
        "declarant_name",
        "bioguide_id",
        "chamber",
        "ticker",
        "company_name",
        "operation_type",
        "transaction_date",
        "disclosure_date",
        "amount_raw",
        "description",
    ]
    payload = "|".join("" if pd.isna(row.get(c, pd.NA)) else str(row.get(c)) for c in cols)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:24]



## Étape 5 — Normalisation Quiver → `final_table`

C'est le cœur du notebook.

L'objectif est de passer d'une table brute Quiver à une table standardisée pour le projet.

### Mapping simplifié des colonnes

| Colonnes possibles côté Quiver | Colonne finale |
|---|---|
| `Name`, `Representative`, `Senator` | `declarant_name` |
| `BioGuideID` | `bioguide_id` |
| `Chamber`, `House` | `chamber` |
| `Party`, `party` | `party` |
| `Ticker` | `ticker` |
| `Company` | `company_name` |
| `TickerType`, `Type` | `asset_type` |
| `Transaction` | `operation_type` |
| `Traded`, `TransactionDate`, `Date` | `transaction_date` |
| `Filed`, `ReportDate` | `disclosure_date` |
| `Trade_Size_USD`, `Range`, `Amount` | `amount_raw` |

### Pourquoi garder les deux dates ?

```text
transaction_date
→ permet de comprendre quand l'élu a réellement tradé.

disclosure_date
→ permet de construire une stratégie réaliste, car c'est la date publique.
```

Le notebook calcule aussi :

```text
disclosure_lag_days
```

Exemple :

```text
transaction_date = 2025-05-15
disclosure_date  = 2025-05-18

disclosure_lag_days = 3
```

### Filtres appliqués à la fin

Le notebook supprime :

```text
- les doublons ;
- les lignes sans ticker ;
- les lignes sans transaction_date ou disclosure_date ;
- les opérations qui ne sont ni Purchase ni Sale.
```

Le but est d'obtenir une table directement utilisable pour la Partie B.


In [5]:

# ============================================================
# 5) NORMALISATION QUIVER → TABLE FINALE
# ============================================================

# Colonnes finales voulues dans final_table.
# Chaque ligne de final_table = une transaction déclarée par un membre du Congrès.
FINAL_COLUMNS = [
    "trade_id",
    "source",
    "declarant_name",
    "bioguide_id",
    "chamber",
    "party",
    "district",
    "state",
    "ticker",
    "company_name",
    "asset_type",
    "asset_type_group",
    "operation_type",
    "operation_group",
    "transaction_date",
    "disclosure_date",
    "disclosure_lag_days",
    "amount_raw",
    "amount_lower_bound",
    "amount_upper_bound",
    "amount_midpoint",
    "amount_method",
    "status",
    "subholding",
    "description",
    "comments",
    "quiver_upload_time",
    "last_modified",
]


def normalize_quiver_congress_trades(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Convertit la réponse brute Quiver en table finale unique.

    Entrée :
        raw_df = DataFrame brut renvoyé par Quiver

    Sortie :
        final_table = DataFrame propre avec les colonnes FINAL_COLUMNS
    """
    if raw_df.empty:
        return pd.DataFrame(columns=FINAL_COLUMNS)

    df = raw_df.copy()
    out = pd.DataFrame(index=df.index)

    # ------------------------------------------------------------
    # 1. Identité du déclarant
    # ------------------------------------------------------------
    # Quiver V2 utilise plutôt Name.
    # Quiver V1 peut utiliser Representative ou Senator.
    out["declarant_name"] = clean_text_series(first_existing(df, ["Name", "Representative", "Senator"]))
    out["bioguide_id"] = clean_text_series(first_existing(df, ["BioGuideID"]))
    out["chamber"] = clean_text_series(first_existing(df, ["Chamber", "House"]))
    out["party"] = clean_text_series(first_existing(df, ["Party", "party"]))
    out["district"] = clean_text_series(first_existing(df, ["District"]))
    out["state"] = clean_text_series(first_existing(df, ["State"]))

    # ------------------------------------------------------------
    # 2. Instrument financier
    # ------------------------------------------------------------
    # ticker = symbole boursier.
    # asset_type_group = type simplifié : Stock, Option, Fund/ETF, etc.
    out["ticker"] = clean_ticker_series(first_existing(df, ["Ticker"]))
    out["company_name"] = clean_text_series(first_existing(df, ["Company"]))
    out["asset_type"] = clean_text_series(first_existing(df, ["TickerType", "Type"]))
    out["asset_type_group"] = out["asset_type"].apply(normalize_asset_type)

    # ------------------------------------------------------------
    # 3. Type d'opération
    # ------------------------------------------------------------
    # operation_type garde la valeur originale.
    # operation_group simplifie en Purchase / Sale / Exchange / Other.
    out["operation_type"] = clean_text_series(first_existing(df, ["Transaction"]))
    out["operation_group"] = out["operation_type"].apply(normalize_operation)

    # ------------------------------------------------------------
    # 4. Dates
    # ------------------------------------------------------------
    # V2 :
    #   Traded = transaction_date
    #   Filed  = disclosure_date
    #
    # V1 :
    #   TransactionDate ou Date = transaction_date
    #   ReportDate             = disclosure_date
    out["transaction_date"] = parse_date_series(first_existing(df, ["Traded", "TransactionDate", "Date"]))
    out["disclosure_date"] = parse_date_series(first_existing(df, ["Filed", "ReportDate"]))

    # Quiver_Upload_Time est conservé si disponible.
    # Il pourra être utile plus tard pour une simulation live plus précise.
    out["quiver_upload_time"] = pd.to_datetime(
        first_existing(df, ["Quiver_Upload_Time"]),
        errors="coerce",
        utc=True,
    ).dt.tz_convert(None)

    out["last_modified"] = pd.to_datetime(
        first_existing(df, ["last_modified"]),
        errors="coerce",
        utc=True,
    ).dt.tz_convert(None)

    # ------------------------------------------------------------
    # 5. Montants
    # ------------------------------------------------------------
    # Quiver V2 donne souvent Trade_Size_USD.
    # Quiver V1 peut donner Range ou Amount.
    out["amount_raw"] = clean_text_series(first_existing(df, ["Trade_Size_USD", "Range", "Amount"]))
    amount_cols = out["amount_raw"].apply(parse_amount_range)
    out = pd.concat([out, amount_cols], axis=1)

    # ------------------------------------------------------------
    # 6. Champs texte complémentaires
    # ------------------------------------------------------------
    # On les garde parce qu'ils peuvent aider à diagnostiquer des cas étranges.
    out["status"] = clean_text_series(first_existing(df, ["Status"]))
    out["subholding"] = clean_text_series(first_existing(df, ["Subholding"]))
    out["description"] = clean_text_series(first_existing(df, ["Description"]))
    out["comments"] = clean_text_series(first_existing(df, ["Comments"]))

    # Source de la donnée.
    out["source"] = "quiver_bulk_congresstrading"

    # ------------------------------------------------------------
    # 7. Délai de divulgation
    # ------------------------------------------------------------
    # Cette colonne répond à : combien de jours entre le trade réel
    # et la divulgation publique ?
    out["disclosure_lag_days"] = (
        out["disclosure_date"] - out["transaction_date"]
    ).dt.days

    # ------------------------------------------------------------
    # 8. Identifiant unique et déduplication
    # ------------------------------------------------------------
    out["trade_id"] = out.apply(make_trade_id, axis=1)
    out = out.drop_duplicates("trade_id").copy()

    # ------------------------------------------------------------
    # 9. Filtres minimaux
    # ------------------------------------------------------------
    # On retire les lignes impossibles à utiliser pour une analyse de signal.
    if DROP_MISSING_TICKER:
        out = out[out["ticker"].notna()].copy()

    if DROP_MISSING_DATES:
        out = out[
            out["transaction_date"].notna()
            & out["disclosure_date"].notna()
        ].copy()

    if KEEP_ONLY_PURCHASE_SALE:
        out = out[out["operation_group"].isin(["Purchase", "Sale"])].copy()

    # ------------------------------------------------------------
    # 10. Ordre final des colonnes et tri
    # ------------------------------------------------------------
    for col in FINAL_COLUMNS:
        if col not in out.columns:
            out[col] = pd.NA

    out = out[FINAL_COLUMNS].sort_values(
        ["disclosure_date", "transaction_date", "declarant_name", "ticker"],
        na_position="last",
    ).reset_index(drop=True)

    return out



## Étape 6 — Exécution

Cette cellule fait réellement tourner la Partie A.

Elle exécute seulement deux lignes importantes :

```python
raw_quiver_table = fetch_all_congress_trades()
final_table = normalize_quiver_congress_trades(raw_quiver_table)
```

Traduction :

```text
1. On télécharge la donnée brute Quiver.
2. On transforme cette donnée brute en table finale propre.
```

La seule sortie affichée est :

```text
final_table
```

C'est volontaire : ce notebook ne crée pas de fichier externe et ne produit pas d'autre table finale.


In [6]:

# ============================================================
# 6) EXÉCUTION — récupération Quiver puis table finale
# ============================================================

# 1. Téléchargement de la donnée brute depuis Quiver.
raw_quiver_table = fetch_all_congress_trades()

# 2. Normalisation vers la table propre du projet.
final_table = normalize_quiver_congress_trades(raw_quiver_table)

# 3. Une seule sortie : la table finale.
display(final_table)


,trade_id,source,declarant_name,bioguide_id,chamber,party,district,state,ticker,company_name,asset_type,asset_type_group,operation_type,operation_group,transaction_date,disclosure_date,disclosure_lag_days,amount_raw,amount_lower_bound,amount_upper_bound,amount_midpoint,amount_method,status,subholding,description,comments,quiver_upload_time,last_modified
0,bbeebccefd740d5b175c876d,quiver_bulk_congresstrading,Mr. Bob Gibbs,G000563,House,Republican,7.0,Ohio,TKR,TIMKEN COMPANY,<NA>,Unknown,Purchase,Purchase,2013-12-18,2014-01-03,16,"$1,001 - $15,000",1001.0,15000.0,8000.5,range_midpoint,NEW,<NA>,<NA>,<NA>,2020-07-26,2023-11-16
1,5f88cb242285140eda378bed,quiver_bulk_congresstrading,Mr. Bob Gibbs,G000563,House,Republican,7.0,Ohio,TEG,"INTEGRYS ENERGY GROUP, INC.",<NA>,Unknown,Sale,Sale,2014-01-03,2014-01-03,0,"$1,001 - $15,000",1001.0,15000.0,8000.5,range_midpoint,NEW,<NA>,<NA>,<NA>,2020-07-26,2023-11-16
2,215ba5ef206f98cd12033cc9,quiver_bulk_congresstrading,Mr. Bill Flores,F000461,House,Republican,17.0,Texas,NGLS,TARGA RESOURCES PARTNERS LP COMMON UNITS REPRE...,<NA>,Unknown,Purchase,Purchase,2013-12-02,2014-01-07,36,"$1,001 - $15,000",1001.0,15000.0,8000.5,range_midpoint,NEW,<NA>,<NA>,<NA>,2020-07-26,2023-11-16
3,01ae688b03323c29c69de794,quiver_bulk_congresstrading,Mr. Bill Flores,F000461,House,Republican,17.0,Texas,BBEP,"BREITBURN ENERGY PARTNERS, L.P. - COMMON UNITS...",<NA>,Unknown,Sale,Sale,2013-12-04,2014-01-07,34,"$250,001 - $500,000",250001.0,500000.0,375000.5,range_midpoint,NEW,<NA>,<NA>,<NA>,2020-07-26,2023-11-16
4,aa633d45b49ce067c24cb5fa,quiver_bulk_congresstrading,Mr. Bill Flores,F000461,House,Republican,17.0,Texas,WMB,"TRANSACTION TYPE.) WILLIAMS COMPANIES, INC.",<NA>,Unknown,Sale,Sale,2013-12-04,2014-01-07,34,"$250,001 - $500,000",250001.0,500000.0,375000.5,range_midpoint,NEW,<NA>,<NA>,<NA>,2020-07-26,2023-11-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99018,a1af92627333516aff7ca592,quiver_bulk_congresstrading,Dwight Evans,E000296,House,Democratic,3.0,Pennsylvania,INTC,INTEL CORPORATION - COMMON STOCK,ST,Stock,Sale,Sale,2026-06-10,2026-06-24,14,"$1,001 - $15,000",1001.0,15000.0,8000.5,range_midpoint,NEW,CETERA,<NA>,<NA>,2026-06-25,2026-06-25
99019,aa714cd03fcfd597d53bb986,quiver_bulk_congresstrading,Greg Stanton,S001211,House,Democratic,4.0,Arizona,TCNNF,TRULIEVE CANNABIS CORP,ST,Stock,Sale,Sale,2026-05-28,2026-06-26,29,"$100,001 - $250,000",100001.0,250000.0,175000.5,range_midpoint,NEW,<NA>,<NA>,<NA>,2026-06-29,2026-06-29
99020,167d41f0c8c07a7c4feaa966,quiver_bulk_congresstrading,Cleo Fields,F000110,House,Democratic,6.0,Louisiana,QNT,QUANTINUUM INC. - CLASS A COMMON STOCK,ST,Stock,Purchase,Purchase,2026-06-04,2026-06-26,22,"$1,001 - $15,000",1001.0,15000.0,8000.5,range_midpoint,NEW,MORGAN STANLEY - E*TRADE #2,<NA>,<NA>,2026-06-29,2026-06-29
99021,3d7adc57392fbe374db01688,quiver_bulk_congresstrading,Cleo Fields,F000110,House,Democratic,6.0,Louisiana,MSFT,MICROSOFT CORPORATION - COMMON STOCK,ST,Stock,Purchase,Purchase,2026-06-11,2026-06-26,15,"$1,001 - $15,000",1001.0,15000.0,8000.5,range_midpoint,NEW,MORGAN STANLEY - E*TRADE #2,<NA>,<NA>,2026-06-29,2026-06-29



# Colonnes de `final_table`

La table finale contient :

```text
trade_id
source
declarant_name
bioguide_id
chamber
party
district
state
ticker
company_name
asset_type
asset_type_group
operation_type
operation_group
transaction_date
disclosure_date
disclosure_lag_days
amount_raw
amount_lower_bound
amount_upper_bound
amount_midpoint
amount_method
status
subholding
description
comments
quiver_upload_time
last_modified
```

Pour la Partie B, les colonnes les plus importantes seront :

```text
declarant_name
bioguide_id
chamber
party
ticker
operation_group
transaction_date
disclosure_date
disclosure_lag_days
amount_midpoint
amount_method
asset_type_group
```

## Ce que cette table permet déjà

Elle permet d'analyser :

```text
- combien de trades sont déclarés ;
- quels élus tradent ;
- quels tickers sont concernés ;
- combien de temps sépare le trade réel de sa divulgation ;
- combien d'achats et de ventes sont disponibles ;
- quelle taille approximative a chaque trade déclaré.
```

## Ce qu'elle ne permet pas encore

Elle ne permet pas encore de calculer une performance, car il manque :

```text
- les prix de marché ;
- les benchmarks SPY / QQQ ;
- les ETF sectoriels ;
- le mapping ticker → secteur ;
- les commissions des élus ;
- les règles de backtest.
```

Donc cette Partie A est suffisante pour la **récupération et la table Quiver**, mais elle n'est pas suffisante pour la stratégie complète.
